In [1]:
import pandas as pd
import re

In [2]:
df = pd.read_csv('B2B_통합_뷰티.csv')

In [3]:
df.head()

,Unnamed: 0,B2B 브랜드 리스트업,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"회사규모 기준\n • Enterprise: 글로벌 대기업 / 메가 브랜드 / 대형 그룹 산하 브랜드\n • Mid-size: 글로벌 유통은 있지만 단일 브랜드 중심, 중견 규모\n • Growth DTC: 빠르게 성장 중인 DTC/세포라·울타 기반 성장 브랜드\n • Indie: 소규모/초기·니치/크리에이터 기반 브랜드"
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,brand,lead_source,inbound_type,contacted,response,country,category,subcategory,scale
2,NaN,amika,outbound,NaN,1,0,US,beauty,Hair,Mid-size
3,NaN,Anastasia Beverly Hills,outbound,NaN,1,0,US,beauty,Makeup,Mid-size
4,NaN,r.e.m beauty(Ariana Grande),outbound,NaN,1,1,US,beauty,Fragrance,Mid-size


# 아웃바운드 환경
“응답률 3%의 B2B 아웃바운드 환경에서  
머신러닝 대신 규칙 기반 스코어링을 선택하여 컨택 우선순위를 재설계한 프로젝트”  

## 데이터 전처리
1️⃣ 문제 정의 (Why)
	•	B2B 아웃바운드 컨택 응답률 3% 미만
	•	무작위 컨택으로 리소스 낭비
	•	ML 적용을 고려했으나 데이터 제약 존재

⸻

2️⃣ 데이터 이해 & 한계 분석 (EDA)
	•	총 198개 브랜드
	•	response=1: 6건
	•	클래스 불균형 심각
	•	ML 학습 불가 판단

👉 “모델을 만들지 않기로 결정”

⸻

3️⃣ 인사이트 도출
	•	Mid-size 브랜드 응답률 최고 (7.1%)
	•	Hair Tools 카테고리 상대적으로 높음
	•	US가 KR 대비 소폭 우세

⸻

4️⃣ 해결 전략

Rule-based Scoring Model 설계

	•	scale, subcategory, country 기반 점수 부여
	•	점수 합산으로 우선순위 정의
	•	신규 브랜드 자동 점수화 가능

⸻

5️⃣ 결과 검증
	•	score ≥ 2:
	•	응답률 ≈ 9.3%
	•	전체 평균 대비 약 3배 개선
	•	컨택 대상 198 → 54로 압축

⸻

6️⃣ 비즈니스 임팩트
	•	컨택 리소스 절감
	•	실무 적용 가능한 자동화 구조
	•	추후 데이터 누적 시 ML 확장 가능

In [4]:
#필요없는 컬럼 삭제
df.drop(columns=['Unnamed: 0'],inplace=True)

df.drop(index=0,inplace=True)

In [5]:
df.head()

,B2B 브랜드 리스트업,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,"회사규모 기준\n • Enterprise: 글로벌 대기업 / 메가 브랜드 / 대형 그룹 산하 브랜드\n • Mid-size: 글로벌 유통은 있지만 단일 브랜드 중심, 중견 규모\n • Growth DTC: 빠르게 성장 중인 DTC/세포라·울타 기반 성장 브랜드\n • Indie: 소규모/초기·니치/크리에이터 기반 브랜드"
1,brand,lead_source,inbound_type,contacted,response,country,category,subcategory,scale
2,amika,outbound,NaN,1,0,US,beauty,Hair,Mid-size
3,Anastasia Beverly Hills,outbound,NaN,1,0,US,beauty,Makeup,Mid-size
4,r.e.m beauty(Ariana Grande),outbound,NaN,1,1,US,beauty,Fragrance,Mid-size
5,Azzaro,outbound,NaN,1,0,US,beauty,Fragrance,Enterprise


In [6]:
#1행(index 0)을 컬럼명으로 선택

# 1. 현재 컬럼 전부 무시하고, index=0 행을 컬럼으로 사용
df.columns = df.iloc[0].tolist()

# 2. 컬럼으로 사용한 행 + 그 위 행 제거
df = df.iloc[1:].reset_index(drop=True)

# 3. 컬럼명 정리
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
)

df.head()

,brand,lead_source,inbound_type,contacted,response,country,category,subcategory,scale
0,amika,outbound,NaN,1,0,US,beauty,Hair,Mid-size
1,Anastasia Beverly Hills,outbound,NaN,1,0,US,beauty,Makeup,Mid-size
2,r.e.m beauty(Ariana Grande),outbound,NaN,1,1,US,beauty,Fragrance,Mid-size
3,Azzaro,outbound,NaN,1,0,US,beauty,Fragrance,Enterprise
4,BaBylissPRO,outbound,NaN,1,0,US,beauty,Hair Tools,Enterprise


In [7]:
df.columns

Index(['brand', 'lead_source', 'inbound_type', 'contacted', 'response',
       'country', 'category', 'subcategory', 'scale'],
      dtype='object')

In [8]:
#결측치 처리
df.isna().sum()
df.dtypes

brand           object
lead_source     object
inbound_type    object
contacted       object
response        object
country         object
category        object
subcategory     object
scale           object
dtype: object

In [9]:
#데이터 타입 변환
df['contacted'] = df['contacted'].astype('int')
df['response'] = df['response'].astype('int')

df.dtypes

brand           object
lead_source     object
inbound_type    object
contacted        int64
response         int64
country         object
category        object
subcategory     object
scale           object
dtype: object

In [10]:
#결측치 처리
df['inbound_type'].isnull().sum()
df['inbound_type'].fillna('NULL', inplace=True)

/var/folders/_s/k3zljh053j76_8t9htqyvxbw0000gn/T/ipykernel_29142/1197974658.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['inbound_type'].fillna('NULL', inplace=True)


In [11]:
#서브카테고리 표준화
df['subcategory'].value_counts()

df['subcategory']=df['subcategory'].str.replace(r'\s*\(.*\)$', '', regex=True)  #괄호 및 괄호 앞 공백 제거
df['subcategory'].value_counts()

subcategory
Skincare                           84
Makeup                             33
Fragrance                          19
Hair / Body                        13
Makeup / Skincare                   8
Hair                                7
Hair Tools                          5
Makeup                              4
Makeup Tools                        3
Hair / body                         3
Beauty Device                       2
Makeup / Skincare                   2
Body / Fragrance                    2
SaaS directory                      1
Tools directory listing             1
Paid reviews / feedback program     1
Developer tools directory           1
3D/AI promo video production        1
Tech blog feature                   1
Reddit marketing campaign           1
Startup directory submissions       1
Investment / venture inquiry        1
Directory listing                   1
Men's Grooming                      1
Skincare / Body                     1
헤어/바디                               1


In [12]:
df['subcategory'].replace('헤어/바디', 'Hair/Body', inplace=True)

/var/folders/_s/k3zljh053j76_8t9htqyvxbw0000gn/T/ipykernel_29142/2060800361.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['subcategory'].replace('헤어/바디', 'Hair/Body', inplace=True)


In [13]:
df.head()

,brand,lead_source,inbound_type,contacted,response,country,category,subcategory,scale
0,amika,outbound,NULL,1,0,US,beauty,Hair,Mid-size
1,Anastasia Beverly Hills,outbound,NULL,1,0,US,beauty,Makeup,Mid-size
2,r.e.m beauty(Ariana Grande),outbound,NULL,1,1,US,beauty,Fragrance,Mid-size
3,Azzaro,outbound,NULL,1,0,US,beauty,Fragrance,Enterprise
4,BaBylissPRO,outbound,NULL,1,0,US,beauty,Hair Tools,Enterprise


In [14]:
#서브카테고리 전처리 - 공백 및 슬래시 정리
df['subcategory'] = (
    df['subcategory']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.replace(' / ', '/', regex=False)
    .str.replace('/ ', '/', regex=False)
    .str.replace(' /', '/', regex=False)
)

#서브카테고리 표준화 매핑 딕셔너리
subcategory_map = {
    # Hair / Body
    'Hair/Body': 'Hair / Body',
    'Hair / body': 'Hair / Body',
    'Hair / Body': 'Hair / Body',
    'Hair/body' : 'Hair / Body',

    # Skincare / Body
    'Skincare/Body': 'Skincare / Body',
    'Skincare / Body': 'Skincare / Body',

    # Makeup / Skincare
    'Makeup/Skincare': 'Makeup / Skincare',
    'Skincare/Makeup': 'Makeup / Skincare',

    # Beauty Device
    'Beauty Device': 'Beauty Device',
    'Beauty Devices': 'Beauty Device',

    # Men's Grooming
    "Men's Grooming / Skincare": "Men's Grooming",
}

#적용
df['subcategory'] = df['subcategory'].replace(subcategory_map)

In [15]:
df['subcategory'].value_counts()


subcategory
Skincare                         85
Makeup                           37
Fragrance                        19
Hair / Body                      17
Makeup / Skincare                11
Hair                              7
Hair Tools                        5
Beauty Device                     3
Makeup Tools                      3
Skincare / Body                   2
Body/Fragrance                    2
Directory listing                 1
Developer tools directory         1
Investment/venture inquiry        1
3D/AI promo video production      1
Startup directory submissions     1
Reddit marketing campaign         1
Tech blog feature                 1
SaaS directory                    1
Men's Grooming/Skincare           1
Paid reviews/feedback program     1
Tools directory listing           1
Men's Grooming                    1
Fragrance/Beauty                  1
Fragrance/Makeup/Skincare         1
Fragrance/Makeup                  1
Makeup/Fragrance                  1
Makeup/Fragrance

In [16]:
import numpy as np

# 1) 'NULL', 'NaN' 문자열도 결측으로 통일 (필요 시)
df['subcategory'] = df['subcategory'].replace(['NULL', 'NaN', 'nan'], np.nan)

# 2) 공백/슬래시 표기 통일: "A / B" "A/ B" "A /B" -> "A/B"
df['subcategory'] = (
    df['subcategory']
    .astype('string')
    .str.strip()
    .str.replace(r'\s*/\s*', '/', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

In [17]:
#서브카테고리 분해
parts = df['subcategory'].str.split('/', expand=True)

df['subcategory_1'] = parts[0].str.strip()
df['subcategory_2'] = parts[1].str.strip() if parts.shape[1] > 1 else np.nan    

In [18]:
df[['subcategory', 'subcategory_1', 'subcategory_2']].head()

,subcategory,subcategory_1,subcategory_2
0,Hair,Hair,<NA>
1,Makeup,Makeup,<NA>
2,Fragrance,Fragrance,<NA>
3,Fragrance,Fragrance,<NA>
4,Hair Tools,Hair Tools,<NA>


In [19]:
df['subcategory_1'].value_counts()

subcategory_1
Skincare                         87
Makeup                           50
Hair                             24
Fragrance                        22
Hair Tools                        5
Makeup Tools                      3
Beauty Device                     3
Men's Grooming                    2
Body                              2
Paid reviews                      1
Tools directory listing           1
Developer tools directory         1
SaaS directory                    1
Tech blog feature                 1
Directory listing                 1
Startup directory submissions     1
3D                                1
Investment                        1
Reddit marketing campaign         1
Short-form promotion              1
Name: count, dtype: Int64

In [20]:
#아웃바운드 전용 subcategory_1
df.loc[df['lead_source'] == 'inbound', 'subcategory_1'] = np.nan

In [21]:
df_out = df[df['lead_source'] == 'outbound'].copy()

In [22]:
df_out.shape

df_out['response'].value_counts()

response
0    192
1      6
Name: count, dtype: int64

이전 target : response를 3종류로 분류   
0 무응답 1 긍정적 회신 2 협업 진행  

-> 현재 모수가 너무 작아 타겟변수를 재정의 할 필요가 있음  
0 무응답 1 회신 이상   

In [23]:
df_out.head()

,brand,lead_source,inbound_type,contacted,response,country,category,subcategory,scale,subcategory_1,subcategory_2
0,amika,outbound,NULL,1,0,US,beauty,Hair,Mid-size,Hair,<NA>
1,Anastasia Beverly Hills,outbound,NULL,1,0,US,beauty,Makeup,Mid-size,Makeup,<NA>
2,r.e.m beauty(Ariana Grande),outbound,NULL,1,1,US,beauty,Fragrance,Mid-size,Fragrance,<NA>
3,Azzaro,outbound,NULL,1,0,US,beauty,Fragrance,Enterprise,Fragrance,<NA>
4,BaBylissPRO,outbound,NULL,1,0,US,beauty,Hair Tools,Enterprise,Hair Tools,<NA>


## EDA

In [24]:
#필요 변수를 선택
# brand 포함해서 feature 정의
features = ['brand', 'country', 'subcategory_1', 'scale']

In [25]:
#평균 응답률
df_out = df_out[features + ['response']].copy()

print(df_out['response'].value_counts())
df_out['response'].mean() #0.0303030303 -> 3.03%

response
0    192
1      6
Name: count, dtype: int64


np.float64(0.030303030303030304)

In [26]:
df_out

,brand,country,subcategory_1,scale,response
0,amika,US,Hair,Mid-size,0
1,Anastasia Beverly Hills,US,Makeup,Mid-size,0
2,r.e.m beauty(Ariana Grande),US,Fragrance,Mid-size,1
3,Azzaro,US,Fragrance,Enterprise,0
4,BaBylissPRO,US,Hair Tools,Enterprise,0
...,...,...,...,...,...
193,BnD,KR,Hair,Indie,0
194,Bodiance,KR,Hair,Indie,0
195,BODY FANTASIES,KR,Fragrance,Enterprise,0
196,BODY HOLIC,KR,Hair,Growth DTC,0


In [27]:
#서브카테고리별 response 집계 및 평균
df_out.groupby('subcategory_1')['response'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False)

,subcategory_1,count,mean
4,Hair Tools,5,0.200000
5,Makeup,50,0.060000
2,Fragrance,22,0.045455
3,Hair,24,0.041667
0,Beauty Device,3,0.000000
1,Body,2,0.000000
6,Makeup Tools,3,0.000000
7,Men's Grooming,2,0.000000
8,Skincare,87,0.000000


-> count : 현재 100사 중에 서브_카테고리별로 전체 몇 개가 분포되어있는지 count된거  
-> mean : 3 / 50 = 0.06% -> 50개중에 3개가 response 1로 응답회신한 비율   
  
  
  
### 인사이트 
- Hair Tool : 표본은 작지만 응답률이 가장 높음 0.2  
- skincare/Makeup : 가장 많이 보냈지만 무반응이거나 응답률 0.06 -> 컨택한 볼륨 대비하여 효율이 낮음   

In [28]:
#scale 공백 제거
df_out['scale'] = df_out['scale'].str.strip()


#브랜드 규모별 응답률
df_out.groupby('scale')['response'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False)

,scale,count,mean
3,Mid-size,28,0.071429
1,Growth DTC,39,0.051282
2,Indie,55,0.018182
0,Enterprise,76,0.013158


sephora, ulta에 입점된 브랜드를 우선적으로 컨택하다보니 enterprise에 해당하는 브랜드가 76개로 가장 많이 컨택하게됨  
enterprise>Indie>GrowthDTC>Mid-size  

응답률의 경우   
mid-size군이 7.1%로 가장 높은 응답률을 보였고  (2위 growth dtc 5%),   
enterprise/indie는 2% 내외로 저조한 성적.

-> mid-size 브랜드는 의사결정 구조가 과도하게 복잡하지 않고 협업 이력이 존재하므로 초기 파트너 타겟으로 선정하여 b2b 컨택을 진행해야함.  

In [29]:
#country
df_out.groupby('country')['response'].agg(['count', 'mean']).reset_index().sort_values(by='mean', ascending=False)

,country,count,mean
1,US,99,0.050505
0,KR,99,0.010101


미국이 5%, 한국이 1%로 미국시장의 응답률이 아주 조금 더 우세한 편.  

# => 머신러닝 예측 모델링을 목표로 하였으나 target의 모수가 너무 작음 이슈가 발생함 
  #### ( 총 198개 중 0 192개 1(응답) 6개)  
  #### (심각한 클래스 불균형)  




## 규칙 기반 스코링 모델(Rule-based Scoring Model)    
### : 과거 데이터에서 관찰된 패턴을 바탕으로 사람이 직접 정의한 규칙에 점수를 부여 -> 우선순위를 수치화


target : response <- 어떤 특징을 가지는 브랜드가 상대적으로 응답 가능성이 높은가?

feature : subcategory_1, scale, country

### 패턴에 따른 규칙 선정
scale == mid-size 일때 +2  
scale == grouwth DTC 일때 +1  
subcategory_1 == hair tools 일때 +1  
country == US 일때 +1  

In [30]:
#점수 매핑 딕셔너리
df_out['score'] = 0

df_out.loc[df_out['scale'] == 'Mid-size', 'score'] += 2
df_out.loc[df_out['scale'] == 'Growth DTC', 'score'] += 1
df_out.loc[df_out['subcategory_1'] == 'Hair Tools', 'score'] += 1
df_out.loc[df_out['country'] == 'US', 'score'] += 1 

In [31]:
#내림차순 결과확인
df_out.sort_values('score', ascending=False)[
    ['score','brand','subcategory_1','scale','country','response']
].head(20)

,score,brand,subcategory_1,scale,country,response
32,4,Drybar,Hair Tools,Mid-size,US,0
10,4,Bio Ionic,Hair Tools,Mid-size,US,0
25,3,Curlsmith,Hair,Mid-size,US,0
39,3,First Aid Beauty,Skincare,Mid-size,US,0
1,3,Anastasia Beverly Hills,Makeup,Mid-size,US,0
45,3,goop,Skincare,Mid-size,US,0
94,3,Peter Thomas Roth,Skincare,Mid-size,US,0
57,3,Jack Black,Men's Grooming,Mid-size,US,0
92,3,PAT McGRATH LABS,Makeup,Mid-size,US,0
0,3,amika,Hair,Mid-size,US,0


In [ ]:
#스코어별 응답률 확인
score_summary = (
    df_out.groupby('score')['response']
    .agg(cnt='count', response_sum='sum', response_rate='mean')
    .reset_index()
    .sort_values('score', ascending=False)
)

score_summary

,score,cnt,response_sum,response_rate
4,4,2,0,0.000000
3,3,20,2,0.100000
2,2,32,3,0.093750
1,1,67,0,0.000000
0,0,77,1,0.012987


score 점수가 2이상일때   
표본 54개 / 응답 5개/ 응답률 약 9.3%   

전체 평균 3.3%에 비하면 약 3배의 응답률을 보임

-> 규칙에 따라 score >=2 일때 우선적으로 컨택할 필요성이 있음  

# 인바운드 분리

-> 근데 이건 진짜 컨택온 리스트들,,,, 분석엔 의미 없습니다

In [32]:
df['inbound_purpose'] = np.where(
    df['lead_source'] == 'inbound',
    df['subcategory_1'],
    np.nan
)